In [ ]:
from Bio.SeqUtils import MeltingTemp as mt
from Bio.SeqUtils import Seq

from oligo_designer_toolsuite.oligo_property_calculator import calc_tm_nn, calc_gc_content

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import lil_matrix

class OligoOverlapCalculatorNew:

    def _get_overlapping_matrix(self, database_region):
        """Creates a matrix that encodes the overlapping of different oligos.
        The matrix has dimensions n_oligos * n_oligos where each row and column
        belong to an oligo. Each entry contains 1 if the correspondent oligos don't
        overlap and 0 if they overlap. This matrix will be used as an adjacency
        matrix, and the sets of non-overlapping oligos are cliques of this graph.

        :param database_region: dictionary containing all the oligos of the region
        :type database_region: dict
        :return: overlapping matrix
        :rtype: pandas.DataFrame
        """

        def _get_overlap(seq1_intervals, seq2_intervals, distance_between_oligos=0):
            for a in seq1_intervals:
                for b in seq2_intervals:
                    if min(a[1], b[1]) - max(a[0], b[0]) >= -distance_between_oligos:
                        return True
            return False

        oligos_indices = list(database_region.keys())  # Keep track of the indices
        intervals = [
            [
                [start[0], end[0]]
                for start, end in zip(
                    database_region[oligo_id]["start"], database_region[oligo_id]["end"]
                )
            ]
            for oligo_id in oligos_indices
        ]

        n_oligos = len(intervals)
        overlapping_matrix = lil_matrix((n_oligos, n_oligos), dtype=int)

        for i in range(n_oligos):
            for j in range(i + 1, n_oligos):
                if _get_overlap(intervals[i], intervals[j]):
                    overlapping_matrix[i, j] = 1

        overlapping_matrix = overlapping_matrix.maximum(overlapping_matrix.transpose())
        overlapping_matrix.setdiag(1)  # Set diagonal elements to 1

        # Create a sparse matrix containing only ones
        ones_matrix = lil_matrix((n_oligos, n_oligos), dtype=int)
        ones_matrix[:, :] = 1

        # Invert the matrix by subtracting the overlapping matrix from the ones matrix
        overlapping_matrix = ones_matrix - overlapping_matrix

        overlapping_matrix = pd.DataFrame(
            data=overlapping_matrix.toarray(),
            columns=oligos_indices,
            index=oligos_indices,
            dtype=int,
        )

        return overlapping_matrix

In [ ]:
class OligoOverlapCalculatorOld:
    def _get_overlapping_matrix(self, database_region: dict, distance_between_oligos: int = 0):
        """Creates a matrix that encodes the overlapping of different oligos. the matrix has dimensions n_oligos * n_oligos where each
        row and column belong to a oligo. Each entry contains 1 if the the correspondent oligos don't overlap and 0 if they overlap, this
        is done because in the next this matrix will be used as an adjacency matrix and the sets of non overlapping oligos are cliques of this graph.

        :pram database_region: dictionary containing all the oligos of the region
        :type database_region: dict
        :return: overlapping matrix
        :rtype: pandas.DataFrame
        """

        def _get_overlap(seq1_intervals, seq2_intervals, distance_between_oligos):
            for a in seq1_intervals:
                for b in seq2_intervals:
                    if min(a[1], b[1]) - max(a[0], b[0]) >= -1 * distance_between_oligos:
                        return True
            return False

        intervals = []
        oligos_indices = []
        
        for oligo_id in database_region.keys():
            interval = []
            oligos_indices.append(oligo_id)  # keep track of the indices
            
            for start, end in zip(database_region[oligo_id]["start"], database_region[oligo_id]["end"]):
                interval.append(
                    [start[0], end[0]]
                )  # save a list of couples of [start,end] of the duplicates of that oligo
            intervals.append(interval)

        overlapping_matrix = np.zeros(
            (len(intervals), len(intervals)), dtype=int
        )  # on the diagonal we have overlaps
        
        for i in range(len(intervals)):
            for j in range(i + 1, len(intervals)):
                if _get_overlap(intervals[i], intervals[j], distance_between_oligos):
                    overlapping_matrix[i, j] = 1
        
        overlapping_matrix = overlapping_matrix + np.transpose(overlapping_matrix) + np.eye(len(intervals))
        overlapping_matrix = np.ones((len(intervals), len(intervals)), dtype=int) - overlapping_matrix
        overlapping_matrix = pd.DataFrame(
            data=overlapping_matrix,
            columns=oligos_indices,
            index=oligos_indices,
            dtype=int,
        )

        return overlapping_matrix

In [ ]:
# Test example
database_region = {
    "oligo1": {
        "start": [[1]],
        "end": [[5]]
    },
    "oligo2": {
        "start": [[6]],
        "end": [[12]]
    },
    "oligo3": {
        "start": [[13]],
        "end": [[35]]
    },
    "oligo4": {
        "start": [[16]],
        "end": [[22]]
    },
    "oligo5": {
        "start": [[16]],
        "end": [[22]]
    },
    "oligo6": {
        "start": [[6]],
        "end": [[12]]
    },
}


calculator = OligoOverlapCalculatorOld()
overlapping_matrix = calculator._get_overlapping_matrix(database_region)
print(overlapping_matrix)

calculator = OligoOverlapCalculatorNew()
overlapping_matrix = calculator._get_overlapping_matrix(database_region)
print(overlapping_matrix)

In [ ]:
import os
import shutil
import unittest

import pandas as pd

from scipy.sparse import csr_matrix
from Bio.SeqUtils import MeltingTemp as mt

from oligo_designer_toolsuite.database import OligoDatabase
from oligo_designer_toolsuite.oligo_efficiency_filter import (
    LowestSetScoring,
    WeightedTmGCOligoScoring,
)
from oligo_designer_toolsuite.oligo_selection import (
    OligosetGeneratorIndependentSet,
    heuristic_selection_independent_set,
)

############################################
# Global Parameters
############################################

FILE_DATABASE = "../data/oligo_selection/oligos_info.tsv"

TM_PARAMETERS = {
    "check": True,  # default
    "strict": True,  # default
    "c_seq": None,  # default
    "shift": 0,  # default
    "nn_table": getattr(mt, "DNA_NN3"),
    "tmm_table": getattr(mt, "DNA_TMM1"),
    "imm_table": getattr(mt, "DNA_IMM1"),
    "de_table": getattr(mt, "DNA_DE1"),
    "dnac1": 50,  # [nM]
    "dnac2": 0,  # [nM]
    "selfcomp": False,  # default
    "saltcorr": 7,  # Owczarzy et al. (2008)
    "Na": 1.25,  # [mM]
    "K": 75,  # [mM]
    "Tris": 20,  # [mM]
    "Mg": 10,  # [mM]
    "dNTPs": 0,  # [mM] default
}

TM_PARAMETERS_CHEM_CORR = {
    "DMSO": 0,  # default
    "fmd": 20,
    "DMSOfactor": 0.75,  # default
    "fmdfactor": 0.65,  # default
    "fmdmethod": 1,  # default
    "GC": None,  # default
}

In [ ]:
tmp_path = "./"
oligo_database = OligoDatabase(
    min_oligos_per_region=2,
    write_regions_with_insufficient_oligos=True,
    dir_output=tmp_path,
)
oligo_database.load_database_from_table(FILE_DATABASE)

oligo_scoring = WeightedTmGCOligoScoring(
    Tm_min=52,
    Tm_opt=60,
    Tm_max=67,
    GC_content_min=40,
    GC_content_opt=50,
    GC_content_max=60,
    Tm_parameters=TM_PARAMETERS,
    Tm_chem_correction_parameters=TM_PARAMETERS_CHEM_CORR,
)
set_scoring = LowestSetScoring(ascending=True)
oligoset_generator = OligosetGeneratorIndependentSet(
    opt_oligoset_size=5,
    min_oligoset_size=2,
    oligos_scoring=oligo_scoring,
    set_scoring=set_scoring,
    heuristic_selection=heuristic_selection_independent_set,
)


In [ ]:
oligo_database = OligoDatabase()
oligo_database.database = {
    "region_1": {
        "A_0": {"oligo": "GCATCTCACTAAGATGTCTGTATCTGCGTGTGCG"},
        "A_1": {"oligo": "AATTAGAAGCGTGTGCGCACATCCCGG"},
        "A_2": {"oligo": "GCATCTCACTAAGATGTCTGTATCTGCGTGTGCGCCCCCACATCC"},
        "A_3": {"oligo": "AAAGCGTGTGTTGTGTTGCGCCCCCACATCCCG"},
        "A_4": {"oligo": "AAAGCTGTTGCGCCCCCACATCC"},
    }
}
oligos, oligo_scores = oligo_scoring.apply(
    oligo_database, region_id="region_1", sequence_type="oligo"
)
index = ["A_0", "A_1", "A_2", "A_3", "A_4"]
data = [
    [0, 1, 1, 0, 0],
    [1, 0, 1, 0, 0],
    [1, 1, 0, 1, 1],
    [0, 0, 1, 0, 1],
    [0, 0, 1, 1, 0],
]
overlapping_matrix = csr_matrix(data)
data = [[0, "A_0", "A_1", "A_2", 1.59, 2.36], [1, "A_4", "A_2", "A_3", 2.15, 4.93]]
true_sets = pd.DataFrame(
    data=data,
    columns=[
        "oligoset_id",
        "oligo_0",
        "oligo_1",
        "oligo_2",
        "set_score_lowest",
        "set_score_sum",
    ],
)

computed_sets = oligoset_generator._get_non_overlapping_sets(
    overlapping_matrix=overlapping_matrix,
    overlapping_matrix_ids=index,
    oligos_scores=oligo_scores,
    n_sets=100,
)
print(computed_sets)
computed_sets["set_score_lowest"] = computed_sets["set_score_lowest"].round(2)
computed_sets["set_score_sum"] = computed_sets["set_score_sum"].round(2)
print(computed_sets)

## Testing Approximation Algorithms for NetworkX

In [ ]:
import numpy as np
import networkx as nx
import random
from scipy.sparse import lil_matrix
import matplotlib.pyplot as plt
from scipy import sparse
import random
import pandas as pd

In [ ]:
# --- Parameters
n_probes = 100
region_length = 1000
oligo_length = 45
distance_between_oligos = 2  # allowed gap threshold

In [ ]:
# --- 1. Generate synthetic (start, end) coordinates for each oligo
probes = []
oligo_scores = {}

for oligo_id in range(n_probes):
    start = random.randint(0, region_length - oligo_length)
    end = start + oligo_length
    probes.append([(start, end)])  # keep your existing structure

    score = random.random()
    oligo_scores[oligo_id] = round(score, 4)

# convert to pandas Series
oligo_scores = pd.Series(oligo_scores, name="oligo_score")

# Inspect
probes[:5], oligo_scores.head()

In [ ]:
# --- 2. Define the overlap check (same as ODT code)
def get_overlap(seq1_intervals, seq2_intervals, distance_between_oligos=distance_between_oligos):
    """Return True if two probes overlap within threshold, False otherwise."""
    return any(
        min(a[1], b[1]) - max(a[0], b[0]) >= -distance_between_oligos
        for a in seq1_intervals
        for b in seq2_intervals
    )

# --- 3. Compute the non-overlap matrix
non_overlap_matrix_ids = [f"oligo{i}" for i in range(n_probes)]
non_overlap_matrix = lil_matrix((n_probes, n_probes), dtype=int)

for i in range(n_probes):
    for j in range(i + 1, n_probes):
        # 1 means no overlap; 0 means overlap
        if not get_overlap(probes[i], probes[j]):
            non_overlap_matrix[i, j] = 1

# Mirror upper triangle to lower triangle
non_overlap_matrix = non_overlap_matrix.maximum(non_overlap_matrix.transpose())

# Set diagonal to 1 (each oligo is compatible with itself)
non_overlap_matrix.setdiag(1)

# Convert to CSR format for fast matrix operations
non_overlap_matrix = non_overlap_matrix.tocsr()

# Count number of 1s and 0s in the matrix
n_total = n_probes * n_probes
n_ones = non_overlap_matrix.count_nonzero()   # efficient for sparse matrices
n_zeros = n_total - n_ones

density = n_ones / n_total

print(f"\nMatrix density summary:")
print(f"  Total entries: {n_total:,}")
print(f"  1s (non-overlaps): {n_ones:,}")
print(f"  0s (overlaps): {n_zeros:,}")
print(f"  Fraction of 1s: {density:.3%}")

In [ ]:
# --- 4. Create a graph from the matrix
G = nx.from_scipy_sparse_array(non_overlap_matrix)
G = nx.relabel_nodes(
            G, {i: non_overlap_matrix_ids[i] for i in range(len(non_overlap_matrix_ids))}
        )
print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

In [ ]:
# --- 6. Try different clique-finding methods
# (Replace these with your tested algorithms)

# 1. Approximation (fast)
try:
    approx = nx.approximation.max_clique(G)
    print(f"nx.approximation.max_clique → {len(approx)}")
except Exception as e:
    print(f"nx.approximation.max_clique failed: {e}")

In [ ]:
# 2. Independent set on complement (safe heuristic)
Gc = nx.complement(G)
indep_clique = nx.algorithms.approximation.maximum_independent_set(Gc)
print(f"max_independent_set complement → {len(indep_clique)}")

In [ ]:
# 3. Greedy custom heuristic
def greedy_max_clique(G):
    nodes = sorted(G.nodes(), key=lambda n: G.degree(n), reverse=True)
    clique = []
    for n in nodes:
        if all(G.has_edge(n, c) for c in clique):
            clique.append(n)
    return clique

greedy = greedy_max_clique(G)
print(greedy)
print(f"greedy_max_clique → {len(greedy)}")

In [ ]:
def greedy_max_clique_new(non_overlap_matrix, non_overlap_matrix_ids):
    degrees = non_overlap_matrix.getnnz(axis=1)
    nodes = np.argsort(-degrees).tolist()
    nodes = np.argsort(-degrees, kind="stable").tolist()
    print(nodes)
    print(degrees[nodes])
    clique = []
    for n in nodes:
        if not clique:
            clique.append(int(n))
        elif non_overlap_matrix[n, clique].getnnz() == len(clique):
            clique.append(int(n))

    return [non_overlap_matrix_ids[c] for c in clique]

greedy = greedy_max_clique_new(non_overlap_matrix, non_overlap_matrix_ids)
print(greedy)
print(f"greedy_max_clique → {len(greedy)}")

In [ ]:
# 4. Exact (slow)
oligoset_size = 25
min_oligoset_size = 5

cliques = nx.algorithms.clique.find_cliques(G)
for clique in cliques:
    if len(clique) > min_oligoset_size:
        largest = clique
    if len(clique) >= oligoset_size:
        print(largest)
        break
print(f"find_cliques (exact) → {len(largest)}")

In [ ]:
print(sorted(list(approx)))
print(sorted(list(indep_clique)))
print(sorted(list(greedy)))
print(sorted(list(largest)))


In [ ]:
def jaccard(a, b):
    return len(a & b) / len(a)

for i, clique in enumerate(all_cliques):
    for j, clique2 in enumerate(all_cliques):
        if i == j:
            continue
        jc = jaccard(set(clique), set(clique2))
        if jc < 0.5:
            print(jc)


In [ ]:
1000//10

In [ ]:
def check_all_ones_dense_friendly(overlap_matrix, indices, ignore_diagonal=True):
    sub = overlap_matrix[np.ix_(indices, indices)]
    if sparse.issparse(sub):
        sub = sub.toarray()

    n = len(indices)
    missing = []
    for i in range(n):
        for j in range(n):
            if ignore_diagonal and i == j:
                continue
            if sub[i, j] != 1:
                missing.append((indices[i], indices[j]))

    return (len(missing) == 0, missing)

all_one, missing = check_all_ones_dense_friendly(non_overlap_matrix, list(approx))
print(all_one, missing)
all_one, missing = check_all_ones_dense_friendly(non_overlap_matrix, list(indep_clique))
print(all_one, missing)
all_one, missing = check_all_ones_dense_friendly(non_overlap_matrix, list(greedy))
print(all_one, missing)
all_one, missing = check_all_ones_dense_friendly(non_overlap_matrix, list(largest))
print(all_one, missing)


In [ ]:
import time
import random
from typing import List, Tuple, Dict, Any
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import networkx as nx


# -----------------------------
# Utilities
# -----------------------------
def build_non_overlap_matrix(intervals: List[Tuple[int, int]]) -> csr_matrix:
    """
    Build a csr_matrix A where A[i,j] == 1 iff intervals[i] and intervals[j] do NOT overlap.
    Diagonal entries are 1.
    """
    n = len(intervals)
    rows, cols, data = [], [], []
    for i in range(n):
        a0, a1 = intervals[i]
        for j in range(n):
            b0, b1 = intervals[j]
            compatible = (a1 <= b0) or (b1 <= a0)  # touching endpoints considered non-overlap
            if compatible:
                rows.append(i)
                cols.append(j)
                data.append(1)
    return csr_matrix((data, (rows, cols)), shape=(n, n))


def is_set_compatible(set_ids: List[str], id_to_idx: Dict[str, int], non_overlap_mat: csr_matrix) -> bool:
    idxs = [id_to_idx[s] for s in set_ids]
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            if non_overlap_mat[idxs[i], idxs[j]] == 0:
                return False
    return True


class SimpleSetScoring:
    """
    Minimal scoring object with the same contract used in earlier code:
    apply(scores_series, set_size) -> (None, {'max_score': value})
    ascending=True means lower is better.
    """
    def __init__(self, ascending=True):
        self.ascending = ascending

    def apply(self, scores_series: pd.Series, set_size: int):
        return None, {'max_score': float(scores_series.max())}


# -----------------------------
# Original heuristic_selection (adapted from user's earlier snippet)
# -----------------------------
def original_heuristic_selection(
    oligoset_init: List[str],
    oligos_scores: pd.Series,
    non_overlap_matrix: csr_matrix,
    non_overlap_matrix_ids: List[str],
    oligoset_size: int,
    heuristic_n_attempts: int,
    set_scoring: SimpleSetScoring,
) -> Tuple[List[str], pd.Series]:
    """
    Original heuristic from the conversation (kept semantics).
    Seeds from the top scored oligos in score order, tries to greedily pick next compatible ones.
    """
    # Sort oligos by score (best first when ascending==True -> smallest first)
    oligo_ids_sorted = oligos_scores.sort_values(ascending=set_scoring.ascending).index.to_list()

    non_overlap_matrix_indices = np.array(
        [non_overlap_matrix_ids.index(oligo_id) for oligo_id in oligo_ids_sorted]
    )
    non_overlap_matrix_sorted = non_overlap_matrix[non_overlap_matrix_indices, :][:, non_overlap_matrix_indices]

    # Initialize max_score with score from initial oligoset
    best_oligoset = oligoset_init
    best_oligoset_max_score = oligos_scores.loc[oligoset_init].max()

    # Try first heuristic_n_attempts seeds (seed is index into oligo_ids_sorted)
    for first_idx in range(min(len(oligo_ids_sorted), heuristic_n_attempts)):
        oligoset_idxs = np.array([first_idx])
        for _ in range(oligoset_size - 1):
            # Find first oligo in sorted array that is not overlapping with any selected oligo
            no_overlap = np.all(non_overlap_matrix_sorted[oligoset_idxs].toarray(), axis=0)
            # Note: original code commented out preventing selecting already selected; enforce it for safety
            no_overlap[oligoset_idxs] = False
            if np.any(no_overlap):
                oligoset_idxs = np.append(oligoset_idxs, np.where(no_overlap)[0][0])
            else:
                break

        if len(oligoset_idxs) == oligoset_size:
            oligoset = [oligo_ids_sorted[idx] for idx in oligoset_idxs]
            oligoset_max_score = oligos_scores.loc[oligoset].max()
            if oligoset_max_score < best_oligoset_max_score:
                best_oligoset_max_score = oligoset_max_score
                best_oligoset = oligoset

    oligos_scores = oligos_scores[oligos_scores <= best_oligoset_max_score]

    return best_oligoset, oligos_scores


# -----------------------------
# Degree-filtered heuristic (improved)
# -----------------------------
def degree_filtered_heuristic_selection(
    oligoset_init: List[str],
    oligos_scores: pd.Series,
    non_overlap_matrix: csr_matrix,
    non_overlap_matrix_ids: List[str],
    oligoset_size: int,
    heuristic_n_attempts: int,
    set_scoring: SimpleSetScoring,
) -> Tuple[List[str], pd.Series]:
    """
    Degree-filtered heuristic:
      - Only seed from oligos with degree >= (oligoset_size - 1)
      - Seed in score order, limited to heuristic_n_attempts seeds
      - Greedily pick the next best-scored compatible oligo
    """
    oligo_ids_sorted = oligos_scores.sort_values(ascending=set_scoring.ascending).index.to_list()
    idx_map = np.array([non_overlap_matrix_ids.index(oligo_id) for oligo_id in oligo_ids_sorted])
    non_overlap_matrix_sorted = non_overlap_matrix[idx_map, :][:, idx_map].tocsr()

    degrees = non_overlap_matrix_sorted.getnnz(axis=1)
    degree_threshold = max(0, oligoset_size - 1)
    eligible_seed_indices = np.where(degrees >= degree_threshold)[0]
    if eligible_seed_indices.size == 0:
        return oligoset_init, oligos_scores

    # seeds in score order but only those eligible
    seed_positions = [i for i in range(len(oligo_ids_sorted)) if i in set(eligible_seed_indices)]
    seed_positions = seed_positions[:heuristic_n_attempts]

    best_oligoset = oligoset_init
    best_oligoset_max_score = oligos_scores.loc[oligoset_init].max() if oligoset_init else float("inf")

    n = non_overlap_matrix_sorted.shape[0]
    for seed_pos in seed_positions:
        oligoset_idxs = [seed_pos]
        for _ in range(oligoset_size - 1):
            compat_matrix = non_overlap_matrix_sorted[oligoset_idxs, :].toarray()
            no_overlap = np.all(compat_matrix, axis=0)
            no_overlap[oligoset_idxs] = False
            candidate_positions = np.where(no_overlap)[0]
            if candidate_positions.size == 0:
                break
            next_pos = int(candidate_positions[0])  # pick top-scored candidate (lowest index)
            oligoset_idxs.append(next_pos)

        if len(oligoset_idxs) == oligoset_size:
            oligoset = [oligo_ids_sorted[idx] for idx in oligoset_idxs]
            oligoset_max_score = oligos_scores.loc[oligoset].max()
            if oligoset_max_score < best_oligoset_max_score:
                best_oligoset_max_score = oligoset_max_score
                best_oligoset = oligoset

    if best_oligoset:
        oligos_scores = oligos_scores[oligos_scores <= best_oligoset_max_score]

    return best_oligoset, oligos_scores


# -----------------------------
# Test harness
# -----------------------------
def test_compare_heuristics(seed: int = 12345):
    random.seed(seed)
    np.random.seed(seed)

    # synthetic pool parameters
    n_oligos = 500
    region_length = 10000
    oligo_length = 25

    # generate random non-overlapping intervals (some overlap inevitable)
    intervals = []
    for _ in range(n_oligos):
        start = random.randint(0, region_length - oligo_length)
        intervals.append((start, start + oligo_length))

    non_overlap_mat = build_non_overlap_matrix(intervals)
    oligo_ids = [f"oligo{i}" for i in range(n_oligos)]

    # random scores in [0,1], rounded to 4 decimals
    scores = np.round(np.random.random(n_oligos), 4)
    oligo_scores = pd.Series(data=scores, index=oligo_ids, name="oligo_score")

    # selection params
    oligoset_size = 100
    heuristic_n_attempts = 200
    set_scoring = SimpleSetScoring(ascending=True)

    # initial set: take best oligos (top oligoset_size by score) if they are compatible else []
    initial_candidates = oligo_scores.sort_values(ascending=True).index.tolist()
    oligoset_init = []
    for candidate in initial_candidates:
        if not oligoset_init:
            oligoset_init.append(candidate)
        else:
            # require compatibility with already chosen
            if is_set_compatible(oligoset_init + [candidate], {k: v for v, k in enumerate(oligo_ids)}, non_overlap_mat):
                oligoset_init.append(candidate)
        if len(oligoset_init) == oligoset_size:
            break

    # run original heuristic
    t0 = time.perf_counter()
    orig_set, orig_scores_pruned = original_heuristic_selection(
        oligoset_init=oligoset_init,
        oligos_scores=oligo_scores.copy(),
        non_overlap_matrix=non_overlap_mat,
        non_overlap_matrix_ids=oligo_ids,
        oligoset_size=oligoset_size,
        heuristic_n_attempts=heuristic_n_attempts,
        set_scoring=set_scoring,
    )
    t1 = time.perf_counter()

    # run degree-filtered heuristic
    t2 = time.perf_counter()
    deg_set, deg_scores_pruned = degree_filtered_heuristic_selection(
        oligoset_init=oligoset_init,
        oligos_scores=oligo_scores.copy(),
        non_overlap_matrix=non_overlap_mat,
        non_overlap_matrix_ids=oligo_ids,
        oligoset_size=oligoset_size,
        heuristic_n_attempts=heuristic_n_attempts,
        set_scoring=set_scoring,
    )
    t3 = time.perf_counter()

    # mapping for compatibility check
    id_to_idx = {k: v for v, k in enumerate(oligo_ids)}

    # Evaluate results
    def eval_result(name: str, chosen: List[str], pruned_scores: pd.Series, elapsed: float):
        print(f"\n[{name}] time: {elapsed:.4f}s")
        if not chosen:
            print(" No set found.")
            return None
        valid = is_set_compatible(chosen, id_to_idx, non_overlap_mat)
        max_score = float(pruned_scores.max()) if (pruned_scores is not None and not pruned_scores.empty) else None
        print(f" Set (size {len(chosen)}): {chosen}")
        print(f" Compatibility valid: {valid}")
        # compute set max from original scores for direct comparison
        set_max_original = oligo_scores.loc[chosen].max()
        print(f" Set max score (original series): {set_max_original:.4f}")
        print(f" Pruned scores max (<= best set max): {max_score}")
        return {'valid': valid, 'set_max': set_max_original}

    res_orig = eval_result("Original heuristic", orig_set, orig_scores_pruned, t1 - t0)
    res_deg = eval_result("Degree-filtered heuristic", deg_set, deg_scores_pruned, t3 - t2)

    # Basic assertions
    if orig_set:
        assert len(orig_set) == oligoset_size, "Original heuristic returned wrong set size"
        assert res_orig['valid'], "Original heuristic returned incompatible set"

    if deg_set:
        assert len(deg_set) == oligoset_size, "Degree-filtered heuristic returned wrong set size"
        assert res_deg['valid'], "Degree-filtered heuristic returned incompatible set"

    # Compare set quality if both found sets
    if orig_set and deg_set:
        print("\nComparison:")
        print(f" Original set max: {res_orig['set_max']:.4f}")
        print(f" Degree-filtered set max: {res_deg['set_max']:.4f}")
        if res_deg['set_max'] <= res_orig['set_max']:
            print(" Degree-filtered heuristic found equal or better (lower) max score.")
        else:
            print(" Original heuristic found better (lower) max score.")

    print("\nTest complete.")


In [ ]:
test_compare_heuristics()

In [ ]:
oligo_database = {"PRR35::2444": {"start": [[1]], "end": [[10]]},
                  "PRR35::2445": {"start": [[5]], "end": [[15]]},
                  "PRR35::3208": {"start": [[15]], "end": [[25]]},
                  "PRR35::3201": {"start": [[25]], "end": [[35]]}}
non_overlap_matrix_ids = ["PRR35::2444", "PRR35::2445", "PRR35::3208", "PRR35::3201"]



intervals = []
for oligo_id in non_overlap_matrix_ids:
    intervals.append([
        [s, e]
        for start, end in zip(
            oligo_database[oligo_id]["start"],
            oligo_database[oligo_id]["end"],
        )
        for s, e in zip(start, end)
    ])


intervals

In [ ]:
from math import inf


distance_between_oligos = 4
def _get_distance(seq1_intervals: list[list[int]], seq2_intervals: list[list[int]]) -> bool:
    distance_min = inf
    for a in seq1_intervals:
        for b in seq2_intervals:
            distance = max(a[0], b[0]) - min(a[1], b[1])
            val = distance if distance > distance_between_oligos else 0
            if val < distance_min:
                distance_min = val
                if distance_min == 0:
                    return 0 
    return distance_min

def _get_distance2(seq1_intervals: list[list[int]], seq2_intervals: list[list[int]]) -> bool:
    distances = min([max(a[0], b[0]) - min(a[1], b[1]) for a in seq1_intervals for b in seq2_intervals])
    if distances <= distance_between_oligos:
        return 0
    return distances

print(_get_distance([[1,15]], [[20,25],[27,35]]))   
print(_get_distance2([[1,15]], [[20,25],[27,35]]))



In [ ]:
from scipy.sparse import csr_matrix, lil_matrix

n_oligos = 4
non_overlap_matrix = lil_matrix((n_oligos, n_oligos), dtype=int)

for i in range(n_oligos):
    for j in range(i + 1, n_oligos):
        non_overlap_matrix[i, j] = _get_distance(intervals[i], intervals[j])

non_overlap_matrix += non_overlap_matrix.T
non_overlap_matrix = non_overlap_matrix.tocsr()


non_overlap_matrix.toarray()

In [ ]:
distance_between_oligos = 0
oligo_database = {"PRR35::2444": {"start": [[1,15]], "end": [[10,25]]},
                  "PRR35::2445": {"start": [[5]], "end": [[15]]},
                  "PRR35::3208": {"start": [[15]], "end": [[25]]},
                  "PRR35::3201": {"start": [[25]], "end": [[35]]}}
non_overlap_matrix_ids = ["PRR35::2444", "PRR35::2445", "PRR35::3208", "PRR35::3201"]

def _get_non_overlap_matrix(oligo_database):


        def _get_overlap(seq1_intervals: list[list[int]], seq2_intervals: list[list[int]]) -> bool:
            # Determine if two ligos overlap based on a distance value
            return any(
                min(a[1], b[1]) - max(a[0], b[0]) >= -distance_between_oligos
                for a in seq1_intervals
                for b in seq2_intervals
            )


        # Get all intervals (start, end)
        intervals = [
            [
                [start[0], end[0]]
                for start, end in zip(
                    oligo_database[oligo_id]["start"],
                    oligo_database[oligo_id]["end"],
                )
            ]
            for oligo_id in non_overlap_matrix_ids
        ]

        # Create a sparse overlap matrix
        n_oligos = len(non_overlap_matrix_ids)
        non_overlap_matrix = lil_matrix((n_oligos, n_oligos), dtype=int)

        # Calculate only upper triangle matrix since the matrix is symmetric
        for i in range(n_oligos):
            for j in range(i + 1, n_oligos):
                if _get_overlap(intervals[i], intervals[j]):
                    non_overlap_matrix[i, j] = 1

        # Fill values of lower triangle
        non_overlap_matrix = non_overlap_matrix.maximum(non_overlap_matrix.transpose())
        # Set diagonal elements to 1 as oligos always overlap with themselves
        non_overlap_matrix.setdiag(1)

        # Create a sparse matrix containing only ones
        ones_matrix = lil_matrix((n_oligos, n_oligos), dtype=int)
        ones_matrix[:, :] = 1

        # Invert theoverlap matrix by subtracting the overlapping matrix from the ones matrix
        non_overlap_matrix = ones_matrix - non_overlap_matrix
        non_overlap_matrix = non_overlap_matrix.tocsr()

        print(non_overlap_matrix.toarray())

_get_non_overlap_matrix(oligo_database)

In [ ]:
def _get_non_overlap_matrix_new(oligo_database):
    def _get_distance(seq1_intervals: list[list[int]], seq2_intervals: list[list[int]]) -> bool:
        # Determine if two ligos do NOT overlap based on a distance value
        distances = min(
            [max(a[0], b[0]) - min(a[1], b[1]) for a in seq1_intervals for b in seq2_intervals]
        )
        if distances <= distance_between_oligos:
            return 0
        return 1  # distances

    # Keep track of the indices

    # Get all intervals (start, end)
    intervals = []
    for oligo_id in non_overlap_matrix_ids:
        intervals.append(
            [
                [s, e]
                # loop through sequences from different genomic regions for the same oligo (can have the same coordinates if coming from shorter and longer exons)
                for start, end in zip(
                    oligo_database[oligo_id]["start"],
                    oligo_database[oligo_id]["end"],
                )
                # loop through exon junction parts for the same oligo
                for s, e in zip(start, end)
            ]
        )

    # Create a sparse non-overlap matrix
    n_oligos = len(non_overlap_matrix_ids)
    non_overlap_matrix = lil_matrix((n_oligos, n_oligos), dtype=int)

    # Calculate only upper triangle matrix since the matrix is symmetric
    for i in range(n_oligos):
        for j in range(i + 1, n_oligos):
            non_overlap_matrix[i, j] = _get_distance(intervals[i], intervals[j])

    # Fill values of lower triangle
    non_overlap_matrix += non_overlap_matrix.T
    non_overlap_matrix = non_overlap_matrix.tocsr()
    print(non_overlap_matrix.toarray())


_get_non_overlap_matrix_new(oligo_database)